# Molecules

A molecular or chemical graph is a structural representation of a molecule in which atoms are nodes and chemical bonds define the edges of the graph. The `kgcnn_torch.molecule` module provides interfaces to construct molecular graphs from chemical structures using backends like [RDKit](https://www.rdkit.org/) or [OpenBabel](http://openbabel.org/).

## Molecular Graph from SMILES

A SMILES string encodes a molecular structure. Using RDKit, we can visualize molecules and extract graph features.

In [ ]:
import rdkit.Chem as Chem

# Caffeine
mol = Chem.MolFromSmiles("CN1C=NC2=C1C(=O)N(C(=O)N2C)C")
mol

## MolGraphInterface

The `kgcnn_torch.molecule.base.MolGraphInterface` defines a unified interface for extracting molecular graphs from chemical informatics backends. The RDKit implementation is available as `MolecularGraphRDKit` (shared with the Keras `kgcnn` package).

> **NOTE**: The molecule module is shared between `kgcnn` (Keras) and `kgcnn_torch`. If you have the full `kgcnn` package installed, you can use `kgcnn.molecule.graph_rdkit.MolecularGraphRDKit` directly.

In [ ]:
# NOTE: MolecularGraphRDKit is shared with the Keras kgcnn package.
# If kgcnn is installed, use:
# from kgcnn.molecule.graph_rdkit import MolecularGraphRDKit
#
# The interface class is defined in kgcnn_torch.molecule.base:
from kgcnn_torch.molecule.base import MolGraphInterface
print("MolGraphInterface methods:")
for attr in dir(MolGraphInterface):
    if not attr.startswith('_'):
        print(f"  {attr}")

## Building Molecular Graphs with RDKit

Here we show how to construct a molecular graph from a SMILES string using RDKit directly, then convert it to a format suitable for GNN training.

> **NOTE**: The following cells require `kgcnn` (Keras) to be installed for `MolecularGraphRDKit`, or you can use RDKit directly as shown.

In [ ]:
import numpy as np
import rdkit.Chem as Chem
from rdkit.Chem import AllChem, Descriptors


def mol_to_graph(smiles: str, make_directed: bool = False):
    """Convert a SMILES string to a graph dictionary using RDKit.
    
    This demonstrates the same operations that MolecularGraphRDKit performs
    under the hood.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Add hydrogens and generate 3D coordinates
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    mol = Chem.RemoveHs(mol)  # Remove H's after conformer generation
    
    # Node features: atomic numbers and symbols
    node_number = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
    node_symbol = [atom.GetSymbol() for atom in mol.GetAtoms()]
    
    # Node coordinates
    conf = mol.GetConformer()
    node_coordinates = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])
    
    # Edge indices and bond types
    edge_indices = []
    edge_number = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = int(bond.GetBondTypeAsDouble())
        edge_indices.append([i, j])
        edge_number.append(bt)
        if make_directed:
            edge_indices.append([j, i])
            edge_number.append(bt)
    
    return {
        "node_number": np.array(node_number),
        "node_symbol": np.array(node_symbol),
        "node_coordinates": np.array(node_coordinates, dtype=np.float32),
        "edge_indices": np.array(edge_indices) if edge_indices else np.zeros((0, 2), dtype=np.int64),
        "edge_number": np.array(edge_number),
        "graph_size": mol.GetNumAtoms(),
    }


# Example: ethanol
graph = mol_to_graph("CCO", make_directed=True)
print("Atoms:", graph["node_symbol"])
print("Atomic numbers:", graph["node_number"])
print("Edge indices:\n", graph["edge_indices"])
print("Bond types:", graph["edge_number"])
print("Coordinates:\n", graph["node_coordinates"])

In [ ]:
# Example: caffeine
graph_caffeine = mol_to_graph("CN1C=NC2=C1C(=O)N(C(=O)N2C)C", make_directed=True)
print("Caffeine:")
print("  Atoms:", graph_caffeine["node_symbol"])
print("  Number of atoms:", graph_caffeine["graph_size"])
print("  Number of bonds:", len(graph_caffeine["edge_number"]) // 2)

## Using MolecularGraphRDKit (from kgcnn)

If `kgcnn` (Keras) is installed, you can use the full `MolecularGraphRDKit` interface which provides richer attribute generation and encoder support.

In [ ]:
# NOTE: This cell requires kgcnn (Keras) to be installed.
# pip install kgcnn

# from kgcnn.molecule.graph_rdkit import MolecularGraphRDKit
#
# mg = MolecularGraphRDKit(make_directed=False)
# mg.from_smiles("CN1C=NC2=C1C(=O)N(C(=O)N2C)C")
# mg.add_hs()
# mg.make_conformer()
# mg.optimize_conformer(force_field="mmff94")
# mg.compute_partial_charges(method="gasteiger")
#
# # Available atom attributes
# print("Atom attributes:", list(MolecularGraphRDKit.atom_fun_dict.keys())[:10], "...")
# print("Bond attributes:", list(MolecularGraphRDKit.bond_fun_dict.keys())[:10], "...")
# print("Mol attributes:", list(MolecularGraphRDKit.mol_fun_dict.keys())[:10], "...")
#
# # Get node attributes with encoder
# from kgcnn_torch.molecule.encoder import OneHotEncoder
# attrs = mg.node_attributes(
#     ["NumBonds", "Mass", "Symbol", "IsInRing"],
#     encoder={"Symbol": OneHotEncoder(["C", "O", "N", "H"], dtype="str")}
# )
# print("First 4 atom attributes:", attrs[:4])

## OneHotEncoder

The `OneHotEncoder` in `kgcnn_torch.molecule.encoder` provides one-hot encoding for categorical atom/bond features.

In [ ]:
from kgcnn_torch.molecule.encoder import OneHotEncoder

# Encode atom symbols
encoder = OneHotEncoder(["C", "N", "O", "F"], dtype="str", add_unknown=True)

print("C:", encoder("C"))    # [1, 0, 0, 0, 0]
print("N:", encoder("N"))    # [0, 1, 0, 0, 0]
print("O:", encoder("O"))    # [0, 0, 1, 0, 0]
print("S:", encoder("S"))    # [0, 0, 0, 0, 1] -- unknown
print("Config:", encoder.get_config())

In [ ]:
# Encode hybridization types
hybr_encoder = OneHotEncoder([2, 3, 4, 5, 6], add_unknown=False)
print("sp3 (3):", hybr_encoder(3))  # [0, 1, 0, 0, 0]
print("sp2 (2):", hybr_encoder(2))  # [1, 0, 0, 0, 0]

## Building Node and Edge Feature Vectors

For GNN training, we typically construct rich feature vectors for atoms and bonds. Here is an example using RDKit directly.

In [ ]:
import rdkit.Chem as Chem
from rdkit.Chem import AllChem


def get_atom_features(atom, symbol_encoder):
    """Generate a feature vector for an atom."""
    features = []
    # One-hot encoded symbol
    features.extend(symbol_encoder(atom.GetSymbol()))
    # Degree
    features.append(atom.GetTotalDegree())
    # Number of Hs
    features.append(atom.GetTotalNumHs())
    # Formal charge
    features.append(atom.GetFormalCharge())
    # Is aromatic
    features.append(int(atom.GetIsAromatic()))
    # Is in ring
    features.append(int(atom.IsInRing()))
    return features


def get_bond_features(bond):
    """Generate a feature vector for a bond."""
    bt = bond.GetBondTypeAsDouble()
    features = [
        int(bt == 1.0),   # Single
        int(bt == 2.0),   # Double
        int(bt == 3.0),   # Triple
        int(bt == 1.5),   # Aromatic
        int(bond.GetIsConjugated()),
        int(bond.IsInRing()),
    ]
    return features


# Example
smiles = "c1ccccc1"  # Benzene
mol = Chem.MolFromSmiles(smiles)
sym_enc = OneHotEncoder(
    ["C", "N", "O", "F", "Cl", "Br", "S", "P"],
    dtype="str", add_unknown=True
)

node_features = [get_atom_features(a, sym_enc) for a in mol.GetAtoms()]
print("Node feature dim:", len(node_features[0]))
print("First atom features:", node_features[0])

edge_features = []
edge_indices = []
for bond in mol.GetBonds():
    i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
    bf = get_bond_features(bond)
    edge_indices.extend([[i, j], [j, i]])
    edge_features.extend([bf, bf])

print("\nEdge feature dim:", len(edge_features[0]))
print("Number of directed edges:", len(edge_features))

## Converting Molecular Graphs to PyG Data

After extracting features, we convert to PyG `Data` objects for use with `kgcnn_torch` models.

In [ ]:
import torch
from torch_geometric.data import Data


def smiles_to_pyg(smiles, label=None, sym_encoder=None):
    """Convert a SMILES string to a PyG Data object with atom/bond features."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    if sym_encoder is None:
        sym_encoder = OneHotEncoder(
            ["C", "N", "O", "F", "Cl", "Br", "S", "P", "I"],
            dtype="str", add_unknown=True
        )
    
    # Node features
    x = torch.tensor(
        [get_atom_features(a, sym_encoder) for a in mol.GetAtoms()],
        dtype=torch.float32
    )
    z = torch.tensor([a.GetAtomicNum() for a in mol.GetAtoms()], dtype=torch.long)
    
    # Edge features
    src, tgt, ef = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = get_bond_features(bond)
        src.extend([i, j])
        tgt.extend([j, i])
        ef.extend([bf, bf])
    
    edge_index = torch.tensor([src, tgt], dtype=torch.long)
    edge_attr = torch.tensor(ef, dtype=torch.float32) if ef else torch.zeros(0, 6)
    
    data = Data(x=x, z=z, edge_index=edge_index, edge_attr=edge_attr)
    if label is not None:
        data.y = torch.tensor([label], dtype=torch.float32)
    
    return data


# Example: convert a few molecules
smiles_list = ["CCO", "CC(=O)O", "c1ccccc1", "CC(C)O"]
labels = [1.0, 2.0, 0.5, 1.5]

pyg_mols = [smiles_to_pyg(s, l) for s, l in zip(smiles_list, labels)]
for s, d in zip(smiles_list, pyg_mols):
    print(f"{s}: {d.num_nodes} atoms, {d.num_edges} edges, x.shape={d.x.shape}")

In [ ]:
# Use with PyG DataLoader
from torch_geometric.loader import DataLoader

loader = DataLoader(pyg_mols, batch_size=2, shuffle=False)
for batch in loader:
    print("Batch:", batch)
    print("  x shape:", batch.x.shape)
    print("  z:", batch.z)
    print("  y:", batch.y)
    print()

## Molecule IO Utilities

The `kgcnn_torch.molecule.io` module provides utilities for reading and writing molecular files (XYZ, SDF, SMILES).

In [ ]:
from kgcnn_torch.molecule.io import (
    read_xyz_file,
    parse_list_to_xyz_str,
    read_mol_list_from_sdf_file,
    read_smiles_file,
)

# Example: create an XYZ string
atoms = ["C", "C", "O"]
coords = [[0.0, 0.0, 0.0], [1.5, 0.0, 0.0], [2.5, 0.5, 0.0]]
xyz_str = parse_list_to_xyz_str([atoms, coords], comment="ethanol")
print("XYZ string:")
print(xyz_str)

## Training Example with Molecular Data

Here is a complete example of training a GCN model on molecular data converted from SMILES.

In [ ]:
from kgcnn_torch.models.gcn import GCNModel
from kgcnn_torch.training.trainer import fit

# Create dataset from SMILES
smiles_data = [
    ("C", -0.5), ("CC", -1.0), ("CCC", -1.5), ("CCCC", -2.0),
    ("CCO", -0.8), ("CCCO", -1.3), ("CCCCO", -1.8),
    ("c1ccccc1", -0.3), ("c1ccc(O)cc1", -0.6), ("c1ccc(N)cc1", -0.4),
    ("CC=O", -0.9), ("CC(=O)O", -1.1), ("CCN", -0.7), ("CCNC", -1.2),
    ("C1CCCCC1", -0.2), ("CC(C)C", -1.8), ("CC(C)O", -1.0), ("CCOCC", -1.6),
    ("c1ccncc1", -0.35), ("c1cc(O)ccc1O", -0.55),
]

enc = OneHotEncoder(["C", "N", "O", "F", "Cl", "Br", "S", "P", "I"], dtype="str", add_unknown=True)
dataset = [smiles_to_pyg(s, l, enc) for s, l in smiles_data]
dataset = [d for d in dataset if d is not None]  # Filter failures

# Note: GCN uses data.x as float features; set edge_weight for adjacency
for d in dataset:
    d.edge_weight = torch.ones(d.edge_index.size(1), 1)

# Split
train_data = dataset[:14]
val_data = dataset[14:]

# GCN model that takes float node features (not embedding)
input_dim = dataset[0].x.size(1)
model = GCNModel(
    node_dim=32, depth=3, gcn_units=32,
    output_units=[16], num_targets=1,
    use_node_embedding=False,  # Use float features, not integer embedding
    node_input_dim=input_dim,
)

history = fit(
    model=model,
    train_loader=DataLoader(train_data, batch_size=8, shuffle=True),
    val_loader=DataLoader(val_data, batch_size=8, shuffle=False),
    optimizer=torch.optim.Adam(model.parameters(), lr=5e-3),
    loss_fn=torch.nn.MSELoss(),
    epochs=50,
    verbose=1,
)

print(f"\nFinal train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss: {history['val_loss'][-1]:.4f}")

> **NOTE**: You can find this page as a Jupyter notebook in the `docs/source` directory of the kgcnn-torch repository.